<a href="https://colab.research.google.com/github/Aleem-mja/DeepLearning-Project-SLIIT-SE4050/blob/IT22313966/IMDB_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# -------------------------------
# 1) Imports, seeds, and globals
# -------------------------------
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix # Added confusion_matrix
import matplotlib.pyplot as plt
import os

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Hyperparameters / globals
MAX_FEATURES = 10000   # number of words to keep (most frequent)
MAXLEN = 500           # cut texts after this number of words (among top MAX_FEATURES)
EMBED_DIM = 128
RNN_UNITS = 64
BATCH_SIZE = 128
EPOCHS = 10
VAL_SAMPLES = 5000     # size of validation set taken from training set
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

In [4]:
# -----------------------------------------
# 2) Data loading and preprocessing (Keras)
# -----------------------------------------
print("Loading IMDB dataset (integer-encoded reviews)...")
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=MAX_FEATURES)

# Pad sequences so all inputs have the same length
x_train = pad_sequences(x_train, maxlen=MAXLEN)
x_test  = pad_sequences(x_test,  maxlen=MAXLEN)

print("Shapes after padding:")
print(f"  x_train: {x_train.shape}, y_train: {y_train.shape}")
print(f"  x_test:  {x_test.shape},  y_test:  {y_test.shape}")

# Create validation set from the end of training set
x_val = x_train[-VAL_SAMPLES:]
y_val = y_train[-VAL_SAMPLES:]
x_train = x_train[:-VAL_SAMPLES]
y_train = y_train[:-VAL_SAMPLES]

print("Shapes after creating validation set:")
print(f"  x_train: {x_train.shape}, x_val: {x_val.shape}, x_test: {x_test.shape}")


Loading IMDB dataset (integer-encoded reviews)...
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Shapes after padding:
  x_train: (25000, 500), y_train: (25000,)
  x_test:  (25000, 500),  y_test:  (25000,)
Shapes after creating validation set:
  x_train: (20000, 500), x_val: (5000, 500), x_test: (25000, 500)


In [19]:
# -----------------------------------------
# 3) Model building (function for reuse)
# -----------------------------------------
def build_rnn_model(max_features=MAX_FEATURES, embed_dim=EMBED_DIM, rnn_units=RNN_UNITS, maxlen=MAXLEN, dropout_rate=0.5):
    """
    Build and return a Sequential RNN model:
      Embedding -> SimpleRNN -> SimpleRNN -> Dropout -> Dense(sigmoid)
    """
    model = Sequential([
        Embedding(max_features, embed_dim, input_length=maxlen),
        SimpleRNN(rnn_units, return_sequences=True),
        SimpleRNN(rnn_units),
        Dropout(dropout_rate),
        Dense(1, activation='sigmoid')
    ])
    return model

# Build and compile
print("Building model...")
model = build_rnn_model()
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


Loading and preprocessing data...


Training data shape: (25000, 500)
Test data shape: (25000, 500)


Epoch 1/25


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


313/313 ━━━━━━━━━━━━━━━━━━━━ 2148s 7s/step - accuracy: 0.6362 - loss: 1.2258 - val_accuracy: 0.8086 - val_loss: 0.6607 - learning_rate: 0.0010
Epoch 2/25
313/313 ━━━━━━━━━━━━━━━━━━━━ 2115s 7s/step - accuracy: 0.8348 - loss: 0.5851 - val_accuracy: 0.8420 - val_loss: 0.5173 - learning_rate: 0.0010
Epoch 3/25
121/313 ━━━━━━━━━━━━━━━━━━━━ 20:08 6s/step - accuracy: 0.8737 - loss: 0.4344

KeyboardInterrupt: 

In [ ]:
# Save the entire model
model.save("imdb_sentiment_model")      # SavedModel format (folder)


In [ ]:
model = load_model("imdb_sentiment_model")


In [23]:
# Evaluate the model
print("Evaluating the model...")
y_pred = model.predict(x_test)
y_pred_classes = (y_pred > 0.5).astype(int).flatten()

Evaluating the model...
782/782 ━━━━━━━━━━━━━━━━━━━━ 220s 281ms/step


Accuracy: 0.5000
Precision: 0.5000
Recall: 1.0000
F1 Score: 0.6667
